When you type `docker run nginx`, a chain of at least four distinct processes springs into action before Nginx ever serves a request. Understanding that chain — who talks to whom, what each component owns, and where the boundaries are — is what separates someone who *uses* Docker from someone who can debug it, secure it, and operate it in production.

In this notebook we'll cover:
- The full Docker stack from CLI to running container
- The Docker client and how it communicates with the daemon
- The Docker daemon (`dockerd`) and what it owns
- `containerd` — the container runtime beneath the daemon
- The OCI specification and `runc` — the lowest level
- The Docker socket and the security implications of who can reach it
- How Docker Desktop works on macOS and Windows

## The Full Stack

Here is the complete call chain that happens when you run a container:

```
You
 │
 │  docker run nginx
 ▼
Docker CLI  (docker)
 │
 │  HTTP POST /containers/create  (REST API over Unix socket)
 ▼
Docker Daemon  (dockerd)
 │
 │  gRPC  (containerd API)
 ▼
containerd
 │
 │  spawns
 ▼
containerd-shim
 │
 │  OCI runtime spec (config.json)
 ▼
runc
 │
 │  clone() syscall — sets up namespaces + cgroups
 ▼
Container process  (nginx)
```

Each layer has a distinct job. The key insight is that `dockerd` itself no longer runs containers — it delegates to `containerd`, which in turn delegates to `runc`. This separation exists so that restarting or upgrading Docker does not kill running containers.

## The Docker Client

The `docker` binary you type commands into is **just a CLI client**. It contains no container logic. Its only job is to translate your commands into HTTP requests and send them to the Docker daemon.

```
docker run --rm -p 8080:80 nginx
     │
     └─► POST /containers/create  { image: "nginx", ... }
         POST /containers/{id}/start
```

The client communicates over the **Docker Engine API** — a versioned REST API documented at `docs.docker.com/engine/api/`. Because it is HTTP, anything that can make HTTP requests can drive Docker: the CLI, Docker Desktop GUI, VS Code's Docker extension, CI/CD pipelines, Python's `docker` SDK, and Kubernetes (historically).

**Useful client commands:**

| Command | What it does |
|---|---|
| `docker version` | Shows client and daemon version, confirms connection |
| `docker info` | Full daemon state: containers, images, drivers, resources |
| `docker system df` | Disk usage by images, containers, volumes |
| `docker context ls` | Shows all daemon endpoints the client can connect to |

> The client and daemon do not have to be on the same machine. Setting `DOCKER_HOST=tcp://remote:2376` points the local `docker` CLI at a remote daemon — useful for deploying to a server without SSHing in.

## The Docker Daemon (`dockerd`)

The Docker daemon is a long-running background process — on Linux it is managed by `systemd` as the `docker` service. It listens for API requests and manages the high-level lifecycle of Docker objects.

**What `dockerd` owns:**

| Responsibility | Detail |
|---|---|
| **Image management** | Pull, push, build, tag, delete images; manages the image store |
| **Container lifecycle** | Create, start, stop, kill, remove containers |
| **Networking** | Creates and manages Docker networks (bridge, overlay, host, none) |
| **Volumes** | Creates and manages named volumes and bind mounts |
| **Build** | Orchestrates `docker build` (or delegates to BuildKit) |
| **Plugins** | Manages network and volume driver plugins |

**Key configuration** — `dockerd` is configured via `/etc/docker/daemon.json` on Linux:

```json
{
  "data-root": "/var/lib/docker",
  "log-driver": "json-file",
  "log-opts": { "max-size": "10m", "max-file": "3" },
  "storage-driver": "overlay2",
  "default-ulimits": { "nofile": { "Hard": 64000, "Soft": 64000 } }
}
```

The daemon stores everything under `/var/lib/docker` by default — images, container filesystems, volumes, and metadata.

## containerd

`containerd` is an **industry-standard container runtime** that sits between `dockerd` and the actual container process. Docker donated it to the **Cloud Native Computing Foundation (CNCF)** in 2017, and it is now the runtime used by Kubernetes, Docker, and most managed Kubernetes services (EKS, GKE, AKS).

The split of responsibilities:

```
dockerd        ← high-level: image building, networking, volumes, API
  │
containerd     ← mid-level: image pull/push/unpack, container create/run/snapshot
  │
runc           ← low-level: set up namespaces/cgroups, exec the process
```

**What `containerd` owns:**
- Pulls and stores image layers from registries
- Unpacks image layers into a container's root filesystem (using snapshots)
- Creates and manages container metadata
- Spawns `containerd-shim` processes — one per container
- Reports container events back to `dockerd`

**Why the shim?** `containerd-shim` is a tiny process that stays alive even when `containerd` restarts. It holds the container's stdin/stdout file descriptors and reports its exit code back. This means upgrading `containerd` does not kill your running containers.

## The OCI Specification and `runc`

In 2015, Docker, CoreOS, and others founded the **Open Container Initiative (OCI)** to standardise two things:

| OCI Spec | What it defines |
|---|---|
| **Image spec** | The format of a container image (layers, manifests, config) |
| **Runtime spec** | How a container runtime must create and run a container from a filesystem bundle |

**`runc`** is Docker's reference implementation of the OCI runtime spec. It is the actual process that makes the kernel calls.

What `runc` does when given an OCI bundle (a root filesystem + `config.json`):
1. Calls `clone()` with namespace flags (`CLONE_NEWPID`, `CLONE_NEWNET`, `CLONE_NEWNS`, …) to create an isolated process tree
2. Sets up `cgroups` to enforce CPU/memory limits
3. Applies seccomp and AppArmor profiles for syscall filtering
4. `exec`s the container's entry point process (e.g., `nginx`)
5. **Exits** — `runc` does not stay running. The container process becomes the child of `containerd-shim`

The OCI standardisation means you can swap `runc` for other runtimes — `gVisor` (Google's sandboxed runtime), `Kata Containers` (VM-based), or `Firecracker` — while keeping `containerd` and the rest of the stack identical.

## The Docker Socket

The Docker daemon listens on a **Unix domain socket** at `/var/run/docker.sock` by default. The Docker CLI connects to this socket to send API requests.

```bash
# These two commands are equivalent:
docker ps
curl --unix-socket /var/run/docker.sock http://localhost/v1.43/containers/json
```

**Why this matters for security:** The Docker socket is a root-equivalent privilege boundary. Any process (or container) that can write to `/var/run/docker.sock` can:
- Start privileged containers
- Mount the host filesystem into a container
- Escape the container and access the host

```
Risk: mounting the socket into a container
─────────────────────────────────────────
docker run -v /var/run/docker.sock:/var/run/docker.sock ...
                    ↑
         Container now has full Docker API access
         = full host access
```

This pattern is sometimes used legitimately (CI agents that need to build Docker images), but it must be understood as granting root on the host. Production workloads should use **rootless Docker** or a socket proxy that restricts which API endpoints are accessible.

## Docker Desktop (macOS and Windows)

Containers are a **Linux kernel feature**. macOS and Windows do not have Linux namespaces or cgroups natively. Docker Desktop solves this by running a lightweight Linux VM in the background.

```
macOS / Windows
  │
  ├── Docker Desktop app (GUI, settings, updates)
  │
  └── LinuxKit VM  (lightweight Alpine-based Linux VM)
        │
        ├── dockerd
        ├── containerd
        └── your containers
```

The `docker` CLI on your Mac or Windows machine tunnels through to the daemon running inside the VM. From your terminal's perspective it is seamless — but it explains a few behaviours:

| Observation | Reason |
|---|---|
| File mounts are slower on Mac than Linux | Bind mounts cross the VM boundary via a virtual filesystem (VirtioFS) |
| `localhost` in a container reaches the Mac host | Docker Desktop injects a special DNS name `host.docker.internal` |
| `/var/run/docker.sock` on Mac is a proxy socket | The real socket lives inside the VM; Docker Desktop proxies it |
| Performance varies with Docker Desktop version | LinuxKit VM and VirtioFS are actively improved each release |

On Linux, there is no VM — the Docker daemon runs directly on the host kernel, and performance is native.

## Hands-On: Inspecting the Engine

The cells below use `docker` CLI commands to observe the architecture at runtime.

In [ ]:
# Client and server version — confirms the client can reach the daemon
# 'Server Engine' shows dockerd; 'containerd' and 'runc' versions are listed separately
!docker version

In [ ]:
# Full daemon state: storage driver, logging driver, cgroup version,
# number of containers and images, kernel version, OS, architecture
!docker info

In [ ]:
# Disk usage breakdown — images, containers, local volumes, build cache
# Shows how much space is reclaimable with 'docker system prune'
!docker system df

In [ ]:
# Start a detached nginx container so we have something to inspect
!docker run -d --name arch-demo nginx:alpine
!docker ps --filter name=arch-demo

In [ ]:
# Full container metadata — everything dockerd knows about this container:
# image layers, network config, mounts, env vars, host config, state
import subprocess, json

raw = subprocess.check_output(["docker", "inspect", "arch-demo"])
data = json.loads(raw)[0]

print("ID         :", data["Id"][:12])
print("Image      :", data["Config"]["Image"])
print("Entrypoint :", data["Config"]["Entrypoint"])
print("Cmd        :", data["Config"]["Cmd"])
print("IP address :", data["NetworkSettings"]["IPAddress"])
print("PID on host:", data["State"]["Pid"])   # the nginx PID as seen by the host OS

In [ ]:
# Processes running inside the container, as seen from outside
# 'docker top' uses the host's /proc — it works because containers share the kernel
!docker top arch-demo

In [ ]:
# Live resource usage snapshot (--no-stream = single sample, not continuous)
# Shows CPU %, memory usage vs limit, network I/O, block I/O — all enforced by cgroups
!docker stats arch-demo --no-stream

In [ ]:
# Show the raw Docker Engine API in action — same call docker CLI makes under the hood
# The socket is at /var/run/docker.sock (Linux) or proxied on Mac via Docker Desktop
!curl -s --unix-socket /var/run/docker.sock \
  'http://localhost/v1.43/containers/arch-demo/json' \
  | python3 -c "import sys,json; d=json.load(sys.stdin); print('Name:', d['Name']); print('Runtime:', d['HostConfig']['Runtime'])"

In [ ]:
# Clean up
!docker rm -f arch-demo

## Summary

- `docker run` triggers a call chain: **CLI → `dockerd` → `containerd` → `containerd-shim` → `runc` → container process**.
- The **Docker client** is a thin HTTP client. It translates CLI commands into REST calls to the Docker Engine API.
- **`dockerd`** owns the high-level lifecycle: images, networks, volumes, and the build system. It communicates with `containerd` over gRPC.
- **`containerd`** handles image storage and unpacking, container creation, and spawning shim processes. It is CNCF-maintained and used directly by Kubernetes.
- **`containerd-shim`** is a per-container process that survives `containerd` restarts and holds the container's I/O.
- **`runc`** is the OCI-compliant runtime that makes the actual kernel calls (`clone`, cgroups, seccomp) and then exits.
- The **Docker socket** (`/var/run/docker.sock`) is a root-equivalent privilege boundary — guard it carefully.
- On **macOS and Windows**, Docker Desktop runs a hidden Linux VM; on Linux, the daemon runs natively on the host kernel.